# Vision Toolkit — Simple Image Processing Notebook

A simplified, notebook version of the Vision Toolkit desktop app.

**Features:**
- Upload an image
- View image info (size, dimensions, channels)
- Grayscale
- Canny edge detection
- Gaussian / Median blur
- Binary / Adaptive threshold
- Drawing tools (rectangle, circle, line, text)
- Save / export the result

*(The webcam capture and histogram bonus features from the original desktop app have been removed to keep this notebook simple.)*


## 1. Setup
Run this cell first to install/import everything needed.

In [ ]:
# If needed, uncomment the line below to install dependencies
# !pip install opencv-python-headless numpy ipywidgets matplotlib

import cv2
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, interactive_output, VBox, HBox, Layout
from IPython.display import display, clear_output
import io
import os


## 2. Upload an image
Use the widget below to upload an image from your computer.

In [ ]:
upload_widget = widgets.FileUpload(
    accept='image/*',
    multiple=False,
    description='Upload Image'
)
display(upload_widget)


In [ ]:
# Run this cell AFTER selecting a file above to load it into OpenCV
original_image = None
file_name = None

def load_uploaded_image():
    global original_image, file_name
    if not upload_widget.value:
        print("No file uploaded yet. Use the widget above first.")
        return
    uploaded_file = list(upload_widget.value.values())[0]
    file_name = uploaded_file['metadata']['name']
    content = uploaded_file['content']
    file_bytes = np.asarray(bytearray(content), dtype=np.uint8)
    original_image = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
    print(f"Loaded '{file_name}' — shape: {original_image.shape}")

load_uploaded_image()


### (Alternative) Load an image from a local file path
If you'd rather not use the upload widget, set a path below and run this cell instead.

In [ ]:
# local_path = "example.jpg"
# original_image = cv2.imread(local_path)
# file_name = os.path.basename(local_path)
# print(f"Loaded '{file_name}' — shape: {original_image.shape}")


## 3. Image information
Basic info about the loaded image.

In [ ]:
def show_image_info(img):
    if img is None:
        print("No image loaded.")
        return
    h, w = img.shape[:2]
    channels = "Grayscale" if len(img.shape) == 2 else f"{img.shape[2]} channels"
    size_kb = "N/A"
    if file_name and os.path.exists(file_name):
        size_kb = f"{os.path.getsize(file_name) / 1024:.1f} KB"
    print(f"File name : {file_name}")
    print(f"Dimensions: {w} x {h} px")
    print(f"Channels  : {channels}")
    print(f"File size : {size_kb}")

show_image_info(original_image)


In [ ]:
def show(img, title="Image"):
    """Display a BGR OpenCV image using matplotlib."""
    if img is None:
        print("No image to display.")
        return
    if len(img.shape) == 2:
        plt.imshow(img, cmap="gray")
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

show(original_image, "Original Image")


## 4. Processing tools

Adjust the controls below and the preview will update automatically. This applies (in order):
Grayscale → Blur → Canny edge detection → Threshold.

In [ ]:
grayscale_chk = widgets.Checkbox(value=False, description='Grayscale')

blur_type = widgets.RadioButtons(
    options=['None', 'Gaussian', 'Median'],
    value='None',
    description='Blur:'
)
blur_amount = widgets.IntSlider(value=1, min=1, max=15, step=1, description='Blur size')

canny_chk = widgets.Checkbox(value=False, description='Canny Edge Detection')
canny_low = widgets.IntSlider(value=50, min=0, max=255, description='Canny low')
canny_high = widgets.IntSlider(value=150, min=0, max=255, description='Canny high')

thresh_type = widgets.RadioButtons(
    options=['None', 'Binary', 'Adaptive'],
    value='None',
    description='Threshold:'
)
thresh_val = widgets.IntSlider(value=128, min=0, max=255, description='Thresh value')

controls = VBox([
    grayscale_chk,
    HBox([blur_type, blur_amount]),
    canny_chk, canny_low, canny_high,
    HBox([thresh_type, thresh_val]),
])

processed_image = None  # holds the latest result of the pipeline

def process_pipeline(grayscale, blur, blur_size, canny, c_low, c_high, thresh, t_val):
    global processed_image
    if original_image is None:
        print("No image loaded. Upload an image first.")
        return
    img = original_image.copy()

    # Grayscale
    if grayscale:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

    # Blur
    ksize = blur_size * 2 + 1  # keep kernel odd
    if blur == 'Gaussian':
        img = cv2.GaussianBlur(img, (ksize, ksize), 0)
    elif blur == 'Median':
        img = cv2.medianBlur(img, ksize)

    # Canny edge detection
    if canny:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        edges = cv2.Canny(gray, c_low, c_high)
        img = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)

    # Threshold
    if thresh == 'Binary':
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        _, th = cv2.threshold(gray, t_val, 255, cv2.THRESH_BINARY)
        img = cv2.cvtColor(th, cv2.COLOR_GRAY2BGR)
    elif thresh == 'Adaptive':
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        th = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)
        img = cv2.cvtColor(th, cv2.COLOR_GRAY2BGR)

    processed_image = img
    show(img, "Processed Image")

out = interactive_output(process_pipeline, {
    'grayscale': grayscale_chk,
    'blur': blur_type,
    'blur_size': blur_amount,
    'canny': canny_chk,
    'c_low': canny_low,
    'c_high': canny_high,
    'thresh': thresh_type,
    't_val': thresh_val,
})

display(controls, out)


## 5. Drawing tools

Draw a rectangle, circle, line, or piece of text on the processed image.
Set the coordinates/options below, then run the "Draw" cell.

Coordinates are in pixels, with `(0, 0)` at the top-left corner of the image.

In [ ]:
shape_type = widgets.Dropdown(
    options=['Rectangle', 'Circle', 'Line', 'Text'],
    value='Rectangle',
    description='Shape:'
)
x1 = widgets.IntText(value=50, description='x1:')
y1 = widgets.IntText(value=50, description='y1:')
x2 = widgets.IntText(value=150, description='x2 / radius:')
y2 = widgets.IntText(value=150, description='y2:')
text_input = widgets.Text(value='Hello', description='Text:')
color_picker = widgets.ColorPicker(value='#e8a15d', description='Color:')
thickness_slider = widgets.IntSlider(value=2, min=1, max=10, description='Thickness')

drawing_controls = VBox([
    shape_type,
    HBox([x1, y1]),
    HBox([x2, y2]),
    text_input,
    HBox([color_picker, thickness_slider]),
])
display(drawing_controls)


In [ ]:
def hex_to_bgr(hex_color):
    hex_color = hex_color.lstrip('#')
    r, g, b = tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    return (b, g, r)

annotated_image = None  # holds the image with drawings on top

def draw_shape():
    global annotated_image
    if processed_image is None:
        print("Run the processing cell above first.")
        return
    base = annotated_image if annotated_image is not None else processed_image
    img = base.copy()
    color = hex_to_bgr(color_picker.value)
    thickness = thickness_slider.value
    p1 = (x1.value, y1.value)
    p2 = (x2.value, y2.value)

    shape = shape_type.value
    if shape == 'Rectangle':
        cv2.rectangle(img, p1, p2, color, thickness)
    elif shape == 'Circle':
        radius = int(np.hypot(x2.value - x1.value, y2.value - y1.value))
        cv2.circle(img, p1, radius, color, thickness)
    elif shape == 'Line':
        cv2.line(img, p1, p2, color, thickness)
    elif shape == 'Text':
        cv2.putText(img, text_input.value, p1, cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, thickness, cv2.LINE_AA)

    annotated_image = img
    show(img, f"{shape} added")

draw_shape()


In [ ]:
# Run this cell to clear all drawings and start again from the processed image
def clear_drawings():
    global annotated_image
    annotated_image = None
    show(processed_image, "Drawings cleared")

clear_drawings()


## 6. Save / export the result

In [ ]:
def save_image(output_path="vision_toolkit_output.png"):
    result = annotated_image if annotated_image is not None else processed_image
    if result is None:
        print("Nothing to save yet — process an image first.")
        return
    ok = cv2.imwrite(output_path, result)
    if ok:
        print(f"Saved to {os.path.abspath(output_path)}")
    else:
        print("Save failed.")

save_image("vision_toolkit_output.png")
